In [3]:
%pip install pandas

  Using cached pandas-3.1.1-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (16 kB)
  Using cached numpy-2.5.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
Using cached pandas-3.1.1-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (12.0 MB)
Using cached numpy-2.5.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.7 MB)
Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd

In [4]:
# Carregar os arquivos
df_dataset = pd.read_csv('../dados/dataset_fullstack_900_smarttv_privacidade.csv')
df_dicionario = pd.read_csv('../dados/dicionario_fullstack_smarttv.csv')

# Obter dimensões do dataset
linhas_dataset, colunas_dataset = df_dataset.shape

# Verificar as informações do dicionário
linhas_dicionario = len(df_dicionario)

print(f"Dataset: {linhas_dataset} linhas, {colunas_dataset} colunas")
print(f"Dicionário: {linhas_dicionario} linhas (representando as 16 colunas)")
print("Colunas do dataset:", list(df_dataset.columns))

Dataset: 900 linhas, 16 colunas
Dicionário: 16 linhas (representando as 16 colunas)
Colunas do dataset: ['id_registro', 'data_hora', 'id_tv', 'categoria_conteudo', 'minutos_visualizados', 'estado_microfone', 'origem_microfone', 'classe_som', 'confianca_classificacao', 'consentimento_coleta', 'modo_privacidade', 'trafego_saida_kb', 'categoria_telemetria', 'anuncios_exibidos', 'interacoes_anuncios', 'regiao']


In [5]:

# 1. Obter o tipo de cada coluna segundo o pandas
dtypes_info = df_dataset.dtypes
print("--- Tipos de Dados ---")
print(dtypes_info)

# 2. Verificar o status da coluna data_hora
print("\n--- Verificação da coluna 'data_hora' ---")
is_datetime = pd.api.types.is_datetime64_any_dtype(df_dataset['data_hora'])
print(f"A coluna 'data_hora' é do tipo datetime? {is_datetime}")
print(f"Tipo real da coluna 'data_hora': {df_dataset['data_hora'].dtype}")

# 3. Verificar se há colunas numéricas carregadas como object (indicativo de lixo)
print("\n--- Verificação de Lixo em Colunas Numéricas ---")
# Colunas que sabemos que são categóricas/texto por natureza
expected_categorical = ['id_registro', 'data_hora', 'id_tv', 'categoria_conteudo', 
                        'estado_microfone', 'origem_microfone', 'classe_som', 
                        'consentimento_coleta', 'modo_privacidade', 'categoria_telemetria', 'regiao']

# Procurar colunas que não estão na lista de categóricas, mas vieram como object
numeric_as_object = [col for col in df_dataset.columns 
                     if df_dataset[col].dtype == 'object' and col not in expected_categorical]

if numeric_as_object:
    print(f"ATENÇÃO: Estas colunas deveriam ser numéricas mas vieram como texto (object): {numeric_as_object}")
else:
    print("Boas notícias: Nenhuma coluna numérica esperada foi carregada como texto (object).")
    
# Mostrar quais colunas foram interpretadas corretamente como números
numeric_columns = [col for col in df_dataset.columns if col not in expected_categorical]
print(f"Colunas reconhecidas corretamente como float/int: {numeric_columns}")

--- Tipos de Dados ---
id_registro                    str
data_hora                      str
id_tv                          str
categoria_conteudo             str
minutos_visualizados       float64
estado_microfone               str
origem_microfone               str
classe_som                     str
confianca_classificacao    float64
consentimento_coleta           str
modo_privacidade               str
trafego_saida_kb           float64
categoria_telemetria           str
anuncios_exibidos            int64
interacoes_anuncios          int64
regiao                         str
dtype: object

--- Verificação da coluna 'data_hora' ---
A coluna 'data_hora' é do tipo datetime? False
Tipo real da coluna 'data_hora': str

--- Verificação de Lixo em Colunas Numéricas ---
Boas notícias: Nenhuma coluna numérica esperada foi carregada como texto (object).
Colunas reconhecidas corretamente como float/int: ['minutos_visualizados', 'confianca_classificacao', 'trafego_saida_kb', 'anuncios_exibidos', 

In [6]:
valores_ausentes = df_dataset.isnull().sum()
percentual_ausentes = (df_dataset.isnull().sum() / len(df_dataset)) * 100

# Criar um DataFrame para exibir os resultados de forma clara
df_ausentes = pd.DataFrame({
    'Valores Ausentes': valores_ausentes,
    'Porcentagem (%)': percentual_ausentes.round(2)
})

# Ordenar pelas colunas com mais valores ausentes para facilitar a leitura
df_ausentes = df_ausentes.sort_values(by='Valores Ausentes', ascending=False)

print(df_ausentes)

                         Valores Ausentes  Porcentagem (%)
confianca_classificacao               635            70.56
minutos_visualizados                   19             2.11
consentimento_coleta                   15             1.67
trafego_saida_kb                       15             1.67
id_tv                                  14             1.56
categoria_conteudo                     12             1.33
id_registro                             0             0.00
data_hora                               0             0.00
classe_som                              0             0.00
origem_microfone                        0             0.00
estado_microfone                        0             0.00
modo_privacidade                        0             0.00
categoria_telemetria                    0             0.00
anuncios_exibidos                       0             0.00
interacoes_anuncios                     0             0.00
regiao                                  0             0.

In [7]:
# 1. Contar linhas totalmente duplicadas (todas as colunas iguais)
duplicadas_inteiras = df_dataset.duplicated().sum()

# 2. Contar linhas duplicadas avaliando apenas a coluna 'id_registro'
duplicadas_id = df_dataset.duplicated(subset=['id_registro']).sum()

print(f"Linhas duplicadas por inteiro: {duplicadas_inteiras}")
print(f"Linhas duplicadas apenas por id_registro: {duplicadas_id}")

Linhas duplicadas por inteiro: 18
Linhas duplicadas apenas por id_registro: 18


In [9]:
df = pd.read_csv('../dados/dataset_fullstack_900_smarttv_privacidade.csv')
# Selecionar as colunas categóricas relevantes
colunas_categoricas = ['categoria_conteudo', 'estado_microfone', 'origem_microfone', 
                       'classe_som', 'consentimento_coleta', 'modo_privacidade', 
                       'categoria_telemetria', 'regiao']

for col in colunas_categoricas:
    print(f"--- {col.upper()} ---")
    print(df[col].value_counts(dropna=False))
    print("\n")

--- CATEGORIA_CONTEUDO ---
categoria_conteudo
Filmes        192
Series        174
Esportes      172
Noticias      126
Musica         67
Infantil       58
Jogos          44
Outros         41
NaN            12
 FILMES         6
ESPOrtes        3
noticias        2
 infantil       2
séries          1
Name: count, dtype: int64


--- ESTADO_MICROFONE ---
estado_microfone
Inativo      548
Ativo        337
ativo          9
 Inativo       4
INATIVO        2
Name: count, dtype: int64


--- ORIGEM_MICROFONE ---
origem_microfone
Nao se aplica            556
TV                       235
Soundbar                  74
Dispositivo Bluetooth     35
Name: count, dtype: int64


--- CLASSE_SOM ---
classe_som
Nao se aplica    556
Silencio          83
Voz               71
Ruido             63
Latido            57
Outro             42
Choro             28
Name: count, dtype: int64


--- CONSENTIMENTO_COLETA ---
consentimento_coleta
Sim              558
Nao              258
Indeterminado     69
NaN            

In [10]:
# Padronizar a coluna estado_microfone para capturar todas as variações 
# (como " Inativo", "INATIVO", "Inativo")
estado_microfone_limpo = df['estado_microfone'].str.strip().str.lower()

# Criar máscaras booleanas para cada inconsistência
mask_minutos = (df['minutos_visualizados'] < 0) | (df['minutos_visualizados'] > 10)
mask_confianca = (df['confianca_classificacao'] < 0) | (df['confianca_classificacao'] > 1)
mask_trafego = df['trafego_saida_kb'] < 0
mask_anuncios_neg = df['anuncios_exibidos'] < 0
mask_interacoes = df['interacoes_anuncios'] > df['anuncios_exibidos']
mask_microfone = (estado_microfone_limpo == 'inativo') & (df['origem_microfone'].str.strip() != 'Nao se aplica')

# Armazenar e exibir as contagens
inconsistencias = {
    'minutos_visualizados fora de 0 a 10': mask_minutos.sum(),
    'confianca_classificacao fora de 0 a 1': mask_confianca.sum(),
    'trafego_saida_kb negativo': mask_trafego.sum(),
    'anuncios_exibidos negativo': mask_anuncios_neg.sum(),
    'interacoes_anuncios > anuncios_exibidos': mask_interacoes.sum(),
    'microfone inativo mas origem_microfone != "Nao se aplica"': mask_microfone.sum()
}

for regra, qtd in inconsistencias.items():
    print(f"{regra}: {qtd} linha(s)")

minutos_visualizados fora de 0 a 10: 11 linha(s)
confianca_classificacao fora de 0 a 1: 5 linha(s)
trafego_saida_kb negativo: 8 linha(s)
anuncios_exibidos negativo: 5 linha(s)
interacoes_anuncios > anuncios_exibidos: 11 linha(s)
microfone inativo mas origem_microfone != "Nao se aplica": 3 linha(s)
